In [1]:
from __future__ import annotations

import numpy.typing as npt
import torch

from cs336_basics.bpe_tokenizer import BpeTokenizer
from cs336_basics.LanguageModel import LanguageModel
from cs336_basics.AdamW import AdamW
from cs336_basics.utils import run_get_batch, run_get_lr_cosine_schedule, run_cross_entropy, run_gradient_clipping, run_save_checkpoint, run_load_checkpoint

import numpy as np
from cs336_basics.pretokenization_example import find_chunk_boundaries
import time

In [2]:
vocab_size = 10000
context_length = 256
d_model = 512
num_layers = 4
num_heads = 16
d_ff = 1344

batch_size=32
device='cpu'
dtype=torch.bfloat16

rope_theta = 10000
betas = [0.9, 0.95]
weight_decay = 1


iterations = 10000

# learning rate hyperparam
max_lr = 1e-2
min_lr = 1e-5
warmup_iters = iterations * 0.2
cosine_cycle_iters = iterations * 0.9
max_l2_norm = 1


In [3]:

# build train data from tokenizer
merges_filepath = 'output/TinyStoriesV2-GPT4-train-heap-merges.json'
vocab_filepath = 'output/TinyStoriesV2-GPT4-train-heap-vocab.json'

print('loading bpe tokenizer')
tokenizer = BpeTokenizer.from_files(vocab_filepath, merges_filepath, ['<|endoftext|>'])

loading bpe tokenizer


In [4]:
len(tokenizer.vocab_dict)

10000

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
lm = LanguageModel(vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta, device=device, dtype=dtype)
lm = lm.to(device)
optimizer = AdamW(lm.parameters(), betas, weight_decay, lr=1e-3, eps=1e-8)


# create data

In [7]:

# input_path = "data/TinyStoriesV2-GPT4-train.txt"
# output_path = "data/train.bin"

# with open(input_path, "rb") as f, open(output_path, "wb") as out:
#     num_chunks = 40

#     boundaries = find_chunk_boundaries(
#         f,
#         num_chunks,
#         b"<|endoftext|>"
#     )

#     chunk = 1
#     for start, end in zip(boundaries[:-1], boundaries[1:]):
#         print(f'=== chunk {chunk} ===')
#         f.seek(start)

#         text = f.read(end - start).decode("utf-8")

#         token_ids = tokenizer.encode(text)

#         np.asarray(
#             token_ids,
#             dtype=np.uint16
#         ).tofile(out)
#         chunk += 1

In [8]:

# input_path = "data/TinyStoriesV2-GPT4-valid.txt"
# output_path = "data/valid.bin"

# with open(input_path, "rb") as f, open(output_path, "wb") as out:
#     num_chunks = 20

#     boundaries = find_chunk_boundaries(
#         f,
#         num_chunks,
#         b"<|endoftext|>"
#     )

#     chunk = 1
#     for start, end in zip(boundaries[:-1], boundaries[1:]):
#         print(f'=== chunk {chunk} ===')
#         f.seek(start)

#         text = f.read(end - start).decode("utf-8")

#         token_ids = tokenizer.encode(text)

#         np.asarray(
#             token_ids,
#             dtype=np.uint16
#         ).tofile(out)
#         chunk += 1

# train

In [9]:
losses = []
train_data = np.memmap(
    "data/train.bin",
    dtype=np.uint16,
    mode="r"
)
# val_data = np.load("valid.npy", mmap_mode="r")
logs = {
    "iteration": [],
    "tokens_processed": [],
    "train_loss": [],
    "val_loss": [],
    "lr": [],
    "iter_time": [],
}

In [ ]:
# lm model
for it in range(iterations):
    print(f'========= itr {it} =========')
    start_time = time.perf_counter()
    optimizer.zero_grad()

    indices, labels = run_get_batch(train_data, batch_size, context_length, device=device)
    # print("embedding shape:", lm.token_embeddings.weight.shape)
    # print("input min:", indices.min().item())
    # print("input max:", indices.max().item())
    # print("dtype:", indices.dtype)
    output = lm.forward(indices, context_length)

    loss = run_cross_entropy(output, labels)
    if (it+1) % 100 == 0:
        print(loss.item())
    # print("indices:", indices.shape, indices.min().item(), indices.max().item())
    # print("labels:", labels.shape, labels.min().item(), labels.max().item())
    # print("vocab:", vocab_size)
    # print("output:", output.shape)
    # print("loss:", loss.item())
    loss.backward()

    lr = run_get_lr_cosine_schedule(it, max_lr, min_lr, warmup_iters, cosine_cycle_iters)
    for group in optimizer.param_groups:
        group["lr"] = lr

    run_gradient_clipping(lm.parameters(), max_l2_norm)
    optimizer.step()

    iter_time = time.perf_counter() - start_time
    tokens_processed = (it + 1) * batch_size * context_length

    logs["iteration"].append(it + 1)
    logs["tokens_processed"].append(tokens_processed)
    logs["train_loss"].append(loss.item())
    logs["lr"].append(lr)
    logs["iter_time"].append(iter_time)    

    if (it+1) % 100 == 0:
        run_save_checkpoint(lm, optimizer, it, f'checkpoints/{it+1}_iteration')


run_save_checkpoint(lm, optimizer, it, 'checkpoints/final')

========= itr 0 =========
indices: torch.Size([32, 256]) 10 9186
labels: (32, 256) 10 9186
vocab: 10000
output: torch.Size([32, 256, 10000])
loss: 9.25
========= itr 1 =========
indices: torch.Size([32, 256]) 10 9341
labels: (32, 256) 10 9341
vocab: 10000
output: torch.Size([32, 256, 10000])
loss: 9.25
========= itr 2 =========
indices: torch.Size([32, 256]) 10 9976
labels: (32, 256) 10 9976
vocab: 10000
output: torch.Size([32, 256, 10000])
loss: 9.25
========= itr 3 =========
indices: torch.Size([32, 256]) 10 9970
labels: (32, 256) 10 9970
vocab: 10000
output: torch.Size([32, 256, 10000])
loss: 9.25
========= itr 4 =========
indices: torch.Size([32, 256]) 10 9949
labels: (32, 256) 10 9949
vocab: 10000
output: torch.Size([32, 256, 10000])
loss: 9.25
========= itr 5 =========
indices: torch.Size([32, 256]) 10 9560
labels: (32, 256) 10 9560
vocab: 10000
output: torch.Size([32, 256, 10000])
loss: 9.25
========= itr 6 =========
indices: torch.Size([32, 256]) 10 9918
labels: (32, 256) 10 99

KeyboardInterrupt: 